## Research question

Can a graph-based event-state model represent higher-order dependence among multiple flood-defence failures and generate plausible multi-breach event sets more consistently than sequential pairwise transition models?

## Motivation

The previous experiments gave me three useful observations:

1. Independence produces highly dispersed synthetic events.
2. A spatial/system dependence kernel produces strongly clustered events.
3. A GNN can learn a dependence score that generates similarly structured events.
4. Both sequential pairwise generators show decreasing completion rates as event size increases.
5. The GNN completion rate declined from 100% for 2-breach events to 51.3% for 5-breach events; the handcrafted kernel declined from 100% to 65.6%.

The last result is the reason for this notebook. I do not want to keep adding complexity to the pairwise transition rule if the underlying representation is still pairwise. Instead, I want to make the current event itself part of the model state, so that the probability of the next breach can depend on the whole set of breaches already selected.

### Important limitation

The current LIWO catalogue does not provide sufficient simultaneous-breach observations to directly estimate true higher-order joint failure probabilities. Therefore, this notebook defines and smoke-tests a research architecture; it does **not** claim a trained empirical higher-order joint-failure model.


## Mathematical formulation

I represent each breach location with a graph embedding

$$ h_i = \operatorname{GNN}(G,X,E)_i $$

For the currently selected event

$$ S_k=\{b_1,\ldots,b_k\}, $$

I construct an event representation

$$ z_k = \operatorname{SetEncoder}\left(\{h_i:i\in S_k\}\right) $$

I then score every candidate \(j\notin S_k\):

$$ s_{j|S_k} = f_\theta(z_k,h_j,e_{S_k,j}) $$

and convert the scores into normalized probabilities:

$$P(b_{k+1}=j\mid S_k)=
\frac{\exp(s_{j\mid S_k}/\tau)}
{\sum_{\ell \notin S_k}\exp(s_{\ell\mid S_k}/\tau)}$$

The important change from the earlier pairwise model is that the conditioning state is the **whole selected event** \(S_k\), not only the most recent breach.


## Proposed higher-order architecture

I structure the proposed model as:

```text
Flood-defence graph
        ↓
Graph encoder
        ↓
Node embeddings h_i
        ↓
Selected breach set
        ↓
Event-state encoder
        ↓
Event embedding z_k
        ↓
Candidate scorer
        ↓
Conditional probability P(next breach | current event)
        ↓
Multi-breach event
```

The candidate scorer can also use the edge/context information between the current event and each unused candidate.

Unlike the earlier pairwise sequential model, the next breach is conditioned on the entire previously generated event state. This is the architectural change I am testing.


## Training-data requirement

A scientifically valid training target for this model would require event-level information containing multiple simultaneous or conditionally dependent defence failures, ideally linked to:

- hydraulic loading;
- breach location;
- defence/system characteristics;
- breach growth;
- hydraulic connectivity;
- resulting flood extent and depth;
- damage consequences.

The current LIWO scenario catalogue is insufficient for direct supervised learning of this target because its scenario structure is predominantly single-breach.

I therefore keep the distinction explicit:

```text
Observed LIWO consequence structure
              ↓
       Current dependence proxies
              ↓
   Higher-order research architecture
```

The architecture below is a proposed next step, not a model trained on fabricated event labels.


## Evaluation framework

The eventual higher-order generator should be compared against four reference approaches:

A. Independence  
B. Handcrafted spatial/system kernel  
C. Sequential pairwise GNN  
D. Higher-order event-state GNN

I would evaluate them along several dimensions.

**Spatial**
- pair-distance distribution
- maximum event diameter
- spatial clustering

**System**
- same-river pair fraction
- same-dijkring pair fraction
- same-system pair fraction

**Consequence**
- log-damage-difference distribution
- consequence similarity

**Generation**
- event completion rate
- diversity of generated events
- sensitivity to event size

**Financial**
- aggregate-loss distribution
- P90
- P95
- P99
- tail exceedance probabilities

The financial metrics would remain proxy-based until the generated events are passed through a physically validated hydraulic consequence model.


## DeepSets-style event encoder

I use a DeepSets-style encoder because I need the event representation to be **permutation-invariant**: the selected breaches form a set, so changing the order in which they are stored should not change the event state.

$$z_k = \rho\left(
\frac{1}{k}\sum_{i\in S_k}\phi(h_i)
\right)$$

This is a simple first choice for representing the whole selected event before scoring the next candidate.


In [2]:

# Notebook 07 — imports


import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

print("PyTorch:", torch.__version__)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

PyTorch: 2.7.1+cu118
Device: cuda


In [3]:

# Higher-order event-state encoder


class EventStateEncoder(nn.Module):

    def __init__(
        self,
        embedding_dim=32,
        hidden_dim=64,
        event_dim=32
    ):
        super().__init__()

        self.phi = nn.Sequential(
            nn.Linear(embedding_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )

        self.rho = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, event_dim),
            nn.ReLU()
        )

    def forward(self, selected_embeddings):

        # selected_embeddings:
        # [event_size, embedding_dim]

        if selected_embeddings.shape[0] == 0:

            return torch.zeros(
                self.rho[-2].out_features,
                device=selected_embeddings.device
            )

        transformed = self.phi(
            selected_embeddings
        )

        pooled = transformed.mean(
            dim=0
        )

        return self.rho(
            pooled
        )

In [ ]:

# Higher-order candidate scorer


class HigherOrderCandidateScorer(nn.Module):

    def __init__(
        self,
        embedding_dim=32,
        event_dim=32,
        edge_dim=3,
        hidden_dim=64
    ):
        super().__init__()

        input_dim = (
            event_dim
            + embedding_dim
            + edge_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                hidden_dim
            ),
            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )

    def forward(
        self,
        event_embedding,
        candidate_embedding,
        edge_features
    ):

        if event_embedding.dim() == 1:
            event_embedding = (
                event_embedding
                .unsqueeze(0)
            )

        if candidate_embedding.dim() == 1:
            candidate_embedding = (
                candidate_embedding
                .unsqueeze(0)
            )

        if edge_features.dim() == 1:
            edge_features = (
                edge_features
                .unsqueeze(0)
            )

        z = torch.cat(
            [
                event_embedding,
                candidate_embedding,
                edge_features
            ],
            dim=1
        )

        return self.mlp(z).squeeze(1)

In [ ]:

# Notebook 07 — self-contained architecture smoke test


# Random node embeddings only for testing tensor shapes.
# These are NOT trained or scientifically meaningful.
torch.manual_seed(2026)

N_TEST_NODES = 10
EMBEDDING_DIM = 32

test_embeddings = torch.randn(
    N_TEST_NODES,
    EMBEDDING_DIM,
    device=device
)

event_encoder = EventStateEncoder(
    embedding_dim=32,
    hidden_dim=64,
    event_dim=32
).to(device)

candidate_scorer = HigherOrderCandidateScorer(
    embedding_dim=32,
    event_dim=32,
    edge_dim=4,
    hidden_dim=64
).to(device)

# Example selected event: 3 breaches
example_nodes = torch.tensor(
    [0, 1, 2],
    dtype=torch.long,
    device=device
)

example_event_embeddings = (
    test_embeddings[example_nodes]
)

example_event_embedding = event_encoder(
    example_event_embeddings
)

# Candidate fourth breach
example_candidate = test_embeddings[3]

example_edge_features = torch.tensor(
    [2.0, 1.0, 0.0, 0.5],
    dtype=torch.float32,
    device=device
)

example_score = candidate_scorer(
    example_event_embedding,
    example_candidate,
    example_edge_features
)

print(
    "Event embedding shape:",
    example_event_embedding.shape
)

print(
    "Candidate score shape:",
    example_score.shape
)

print(
    "Candidate score:",
    example_score.item()
)

Event embedding shape: torch.Size([32])
Candidate score shape: torch.Size([1])
Candidate score: -0.12152647227048874


In [7]:

# Higher-order candidate probability mechanism


def candidate_probabilities(
    event_embedding,
    candidate_embeddings,
    candidate_edge_features,
    scorer,
    temperature=0.10
):
    """
    Convert candidate scores into a probability distribution.

    Parameters
    ----------
    event_embedding : [event_dim]
    candidate_embeddings : [n_candidates, embedding_dim]
    candidate_edge_features : [n_candidates, edge_dim]
    scorer : HigherOrderCandidateScorer
    temperature : softmax temperature
    """

    n_candidates = candidate_embeddings.shape[0]

    event_batch = event_embedding.unsqueeze(0).repeat(
        n_candidates,
        1
    )

    scores = scorer(
        event_batch,
        candidate_embeddings,
        candidate_edge_features
    )

    probabilities = torch.softmax(
        scores / temperature,
        dim=0
    )

    return scores, probabilities

In [ ]:

# Probability normalization test


torch.manual_seed(2026)

n_candidates = 7

candidate_embeddings = torch.randn(
    n_candidates,
    32,
    device=device
)

candidate_edge_features = torch.randn(
    n_candidates,
    4,
    device=device
)

scores, probabilities = candidate_probabilities(
    example_event_embedding,
    candidate_embeddings,
    candidate_edge_features,
    candidate_scorer,
    temperature=0.10
)

print("Scores shape:", scores.shape)
print("Probabilities shape:", probabilities.shape)
print("Probability sum:", probabilities.sum().item())
print("Minimum probability:", probabilities.min().item())
print("Maximum probability:", probabilities.max().item())

Scores shape: torch.Size([7])
Probabilities shape: torch.Size([7])
Probability sum: 1.0
Minimum probability: 0.09240029752254486
Maximum probability: 0.21660976111888885


In [9]:

# Exclude already-selected locations


candidate_ids = np.array([
    101, 102, 103, 104, 105, 106, 107
])

selected_ids = {
    102,
    105
}

available_mask = np.array([
    cid not in selected_ids
    for cid in candidate_ids
])

available_ids = candidate_ids[
    available_mask
]

available_embeddings = candidate_embeddings[
    torch.tensor(
        available_mask,
        dtype=torch.bool,
        device=device
    )
]

available_edge_features = candidate_edge_features[
    torch.tensor(
        available_mask,
        dtype=torch.bool,
        device=device
    )
]

scores_available, probs_available = (
    candidate_probabilities(
        example_event_embedding,
        available_embeddings,
        available_edge_features,
        candidate_scorer,
        temperature=0.10
    )
)

print("Original candidates:", candidate_ids)
print("Selected:", sorted(selected_ids))
print("Available:", available_ids)
print(
    "Probability sum:",
    probs_available.sum().item()
)

Original candidates: [101 102 103 104 105 106 107]
Selected: [102, 105]
Available: [101 103 104 106 107]
Probability sum: 1.0


## Proposed training objective

For an observed or hydraulically simulated multi-breach event

$$S=\{b_1,b_2,\dots,b_m\},$$

the model can be trained autoregressively:

$$P(b_1)$$
$$P(b_2\mid b_1)$$
$$P(b_3\mid b_1,b_2)$$
$$\dots$$
$$P(b_m\mid b_1,\dots,b_{m-1})$$

with the negative log-likelihood objective

$$L=-\sum_k\log P(b_k\mid S_{k-1},H)$$

where \(H\) represents hazard/loading and system context.

The important difference from the current pairwise generator is that \(S_{k-1}\) can contain the **whole event history**.

### Required training data

A valid training dataset should contain multi-breach event realizations or high-fidelity hydraulic simulations with:

- breach/defence locations;
- hydraulic loading;
- breach initiation and growth;
- defence/system relationships;
- spatial connectivity;
- resulting flood extent/depth;
- damage consequences.

The current LIWO scenario catalogue is useful for structural exploration and prototype development, but it is insufficient for direct supervised estimation of higher-order joint failure probabilities.

### Validation

The higher-order generator should eventually be evaluated against held-out observed or hydraulically simulated events using:

- event completion probability;
- spatial-distance distributions;
- system-consistency statistics;
- flood-depth/extent similarity;
- damage-distribution similarity;
- aggregate-loss tail metrics;
- calibration and uncertainty assessment.

I would compare the higher-order model with the same independence, handcrafted-kernel, and sequential-GNN baselines used in the earlier experiments.


## BREACH-AI architecture

The proposed end-to-end structure is:

```text
Hazard / loading
        ↓
Flood-defence graph
        ↓
Graph neural encoder
        ↓
Node representations
        ↓
Selected breach set
        ↓
Permutation-invariant event-state encoder
        ↓
Candidate breach scorer
        ↓
P(next breach | current event)
        ↓
Multi-breach event generator
        ↓
Hydraulic consequence model / surrogate
        ↓
Damage and financial loss
        ↓
Climate-state conditioning
        ↓
Insurability assessment
```

This is the direction I want to take beyond the current prototype: use the learned spatial/system representation to generate richer events, but ultimately constrain and validate those events through hydraulic consequences before using them for financial-risk and insurability analysis.
